# space-awareness-ml: run everything

Runtime > Change runtime type > T4 GPU, then run all. Every stage skips itself if its output already exists, so after a disconnect just run all again. Results live on Google Drive in `MyDrive/space-awareness-ml/results`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from pathlib import Path
from google.colab import drive

REPO_URL = "https://github.com/YOUR_USER/space-awareness-ml.git"
REPO = Path("/content/space-awareness-ml")
DATA = Path("/content/data")

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/space-awareness-ml")
(DRIVE / "results").mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE / "cache").mkdir(parents=True, exist_ok=True)
DATA.mkdir(exist_ok=True)

In [ ]:
if REPO.exists():
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}

# results/ inside the repo points at Drive so every script writes straight to persistent storage
res = REPO / "results"
if res.exists() and not res.is_symlink():
    !cp -rn {res}/. {DRIVE}/results/ && rm -rf {res}
if not res.exists():
    res.symlink_to(DRIVE / "results")

%cd {REPO}
!pip install -q -r requirements.txt

In [ ]:
zip_path = DRIVE / "cache" / "DUST.zip"
if not zip_path.exists():
    !wget -q --show-progress -O {zip_path}.part "https://zenodo.org/records/20255672/files/DUST.zip?download=1" && mv {zip_path}.part {zip_path}
if not (DATA / "DUST" / "SET 1").exists():
    !unzip -q {zip_path} -d {DATA}
code_zip = DRIVE / "cache" / "DUST_code.zip"
if not code_zip.exists():
    !wget -q -O {code_zip} "https://zenodo.org/records/20255672/files/DUST%20code.zip?download=1"

print("DUST tree, directories to depth 3, with file counts:")
for d in sorted(p for p in (DATA / "DUST").glob("*/*/*") if p.is_dir()):
    print(f"{len(list(d.iterdir())):5d}  {d.relative_to(DATA)}")

In [ ]:
from huggingface_hub import snapshot_download

# Only the magnitude files are needed: the time grid is a fixed 0.1 s step (10,000 points) in every file.
ROBO6 = Path(snapshot_download("kyselica/RoBo6", repo_type="dataset", local_dir=DATA / "robo6",
                               allow_patterns=["*.csv", "README.md", "data/*/*_mag.txt"]))
print(ROBO6, sum(1 for _ in ROBO6.glob("data/*/*_mag.txt")), "mag files")

## Project A: DUST prep

In [ ]:
A_DATA = DATA / "dust_yolo"
if not (A_DATA / "gt.json").exists():
    !python a_dust/prep.py --dust {DATA}/DUST --out {A_DATA} --results results
else:
    print("prep already done")

In [ ]:
# CHECKPOINT A1: copy the tree printed above, this JSON and the figure back for review
!python a_dust/figures.py --data {A_DATA} --only method
print(open("results/prep_stats.json").read())
from IPython.display import Image, display
display(Image("results/figures/a_method.png"))

## Training, evaluation and Project B

These cells are added after checkpoint A1 is reviewed.

In [ ]:
# In Cursor: Colab sidebar > /content > right-click results.zip > Download, save it in the local repo root
!rm -f /content/results.zip && cd {DRIVE} && zip -qr /content/results.zip results && ls -lh /content/results.zip